# Step 4: Wav2Vec2 Audio Deepfake Model Architecture

- **Backbone:** Pre-trained `facebook/wav2vec2-base` from Hugging Face
- **Freezing:** CNN raw feature extractor frozen; top 2 transformer layers fine-tuned
- **Pooling:** Temporal mean pooling across time steps
- **Classification Head:** LayerNorm $\rightarrow$ Linear(768, 256) $\rightarrow$ GELU $\rightarrow$ Dropout(0.3) $\rightarrow$ Linear(256, 2)

In [1]:
import torch
import torch.nn as nn
from transformers import Wav2Vec2Model

class AudioDeepfakeClassifier(nn.Module):
    def __init__(
        self,
        model_name_or_path: str = "facebook/wav2vec2-base",
        num_classes: int = 2,
        hidden_dim: int = 256,
        dropout_rate: float = 0.3,
        freeze_feature_extractor: bool = True,
        unfreeze_top_transformer_layers: int = 2
    ):
        super().__init__()
        self.wav2vec2 = Wav2Vec2Model.from_pretrained(model_name_or_path)

        # Freeze feature extractor
        if freeze_feature_extractor and hasattr(self.wav2vec2, "feature_extractor"):
            for param in self.wav2vec2.feature_extractor.parameters():
                param.requires_grad = False

        # Layer freezing
        total_layers = len(self.wav2vec2.encoder.layers)
        freeze_until = total_layers - unfreeze_top_transformer_layers
        for i, layer in enumerate(self.wav2vec2.encoder.layers):
            if i < freeze_until:
                for param in layer.parameters():
                    param.requires_grad = False
            else:
                for param in layer.parameters():
                    param.requires_grad = True

        backbone_dim = self.wav2vec2.config.hidden_size  # 768

        # Classification Head
        self.classifier = nn.Sequential(
            nn.LayerNorm(backbone_dim),
            nn.Linear(backbone_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim, num_classes)
        )
        self.loss_fn = nn.CrossEntropyLoss()

    def forward(self, input_values: torch.Tensor, labels: torch.Tensor = None):
        outputs = self.wav2vec2(input_values=input_values)
        hidden_states = outputs.last_hidden_state  # (B, T, 768)
        pooled = torch.mean(hidden_states, dim=1)  # (B, 768)
        logits = self.classifier(pooled)
        probs = torch.softmax(logits, dim=-1)

        res = {"logits": logits, "probs": probs}
        if labels is not None:
            res["loss"] = self.loss_fn(logits, labels)
        return res


e:\Deepfake\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Instantiate and Test Model with Dummy Input
device = "cuda" if torch.cuda.is_available() else "cpu"
model = AudioDeepfakeClassifier().to(device)

dummy_input = torch.randn(2, 64000).to(device)
dummy_labels = torch.tensor([0, 1]).to(device)

out = model(dummy_input, labels=dummy_labels)
print(f"Forward Pass Successful!")
print(f"Logits Shape: {out['logits'].shape}")
print(f"Probs Shape:  {out['probs'].shape}")
print(f"Sample Loss:  {out['loss'].item():.4f}")


Loading weights: 100%|██████████| 211/211 [00:00<00:00, 33214.42it/s]
[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_hid.weight           | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Forward Pass Successful!
Logits Shape: torch.Size([2, 2])
Probs Shape:  torch.Size([2, 2])
Sample Loss:  0.7465
